# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [4]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Rows: {len(df)}")
print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units: {total_units:,}")

print("The 400 orders brought in $8,520.00 from 783 units sold, which is ~$10.88 per unit & $21.30 per order.")

Rows: 400
Total revenue: $8,520.00
Total units: 783
The 400 orders brought in $8,520.00 from 783 units sold, which is ~$10.88 per unit & $21.30 per order.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = (
    df.groupby('category', as_index=False)['revenue'].sum()
      .sort_values('revenue', ascending=False)
      .reset_index(drop=True)
)
by_category['share_pct'] = (by_category['revenue'] / df['revenue'].sum() * 100).round(1)
by_category

print("Food is half of all revenue ($4,293.00, 50.4%); merch is second at 20.8%; drink is third at 18.2%; RainGear is the smallest at 10.6%.")

Food is half of all revenue ($4,293.00, 50.4%); merch is second at 20.8%; drink is third at 18.2%; RainGear is the smallest at 10.6%.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [5]:
by_vendor = (
    df.groupby('vendor_id')['revenue']
      .agg(avg_order_revenue='mean', orders='count', std='std')
      .sort_values('avg_order_revenue', ascending=False)
)
# standard error of each vendor's average to see how much the ranking can be trusted
by_vendor['std_error'] = by_vendor['std'] / np.sqrt(by_vendor['orders'])
by_vendor.round(2)

print("V-01 has the highest average order revenue at $22.60 across 94 orders, but every vendor has 93 to 108 orders, so none of these averages is on a tiny group. However, the gap from first (V-01, $22.60) to last (V-10, $20.31) is only about $2.28, while each average has a standard error of ~$1.6 to $1.8 - meaning therankingis mostly noise.")

V-01 has the highest average order revenue at $22.60 across 94 orders, but every vendor has 93 to 108 orders, so none of these averages is on a tiny group. However, the gap from first (V-01, $22.60) to last (V-10, $20.31) is only about $2.28, while each average has a standard error of ~$1.6 to $1.8 - meaning therankingis mostly noise.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [6]:
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()
merch_share = merch_revenue / df['revenue'].sum() * 100
print(f"Merch share of revenue: {merch_share:.1f}%")

print("Merch brings in 20.8% of revenue ($1,771.50)--> one dollar in every five, & it matches Q2.")

Merch share of revenue: 20.8%
Merch brings in 20.8% of revenue ($1,771.50)--> one dollar in every five, & it matches Q2.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [7]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
# finding the vendor id that is in the orders but missing from the lookup
missing_ids = sorted(set(df['vendor_id']) - set(vendor_names['vendor_id']))
print("Vendor ids with no name:", missing_ids)

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

# prove the merge did not add, drop, or change anything
assert len(joined) == len(df), 'row count changed'
assert abs(joined['revenue'].sum() - df['revenue'].sum()) < 0.01, 'revenue changed'
print(f"Rows before/after: {len(df)} / {len(joined)}")
print(f"Revenue before/after: ${df['revenue'].sum():,.2f} / ${joined['revenue'].sum():,.2f}")

# size of unmatched group
unmatched = joined[joined['vendor_name'].isna()]
print(f"Unmatched orders: {len(unmatched)} ({len(unmatched)/len(joined):.1%} of orders)")
print(f"Unmatched revenue: ${unmatched['revenue'].sum():,.2f} ({unmatched['revenue'].sum()/joined['revenue'].sum():.1%} of revenue)")

# keep the rows, labeling to make sure they show up in reports
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown (' + joined['vendor_id'] + ')')
joined.head()

print("V-18 is in the orders but not in the lookup table. It has 108 orders and $2,349.00 in revenue, which is 27.0% of orders and 27.6% of all revenue, therefore not a rounding error. I kept the rows with the left join instead of dropping them, and labeled them `Unknown (V-18)` so they still show up in every total (didn't guess name). The lookup table's owner needs to add it, and until then anything I report about V-18 should be flagged as unverified.")

Vendor ids with no name: ['V-18']
Rows before/after: 400 / 400
Revenue before/after: $8,520.00 / $8,520.00
Unmatched orders: 108 (27.0% of orders)
Unmatched revenue: $2,349.00 (27.6% of revenue)
V-18 is in the orders but not in the lookup table. It has 108 orders and $2,349.00 in revenue, which is 27.0% of orders and 27.6% of all revenue, therefore not a rounding error. I kept the rows with the left join instead of dropping them, and labeled them `Unknown (V-18)` so they still show up in every total (didn't guess name). The lookup table's owner needs to add it, and until then anything I report about V-18 should be flagged as unverified.


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [8]:
pivot = joined.pivot_table(
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total',
)
print(pivot)
print("The bottom-right($8,520.00) matches the Q1 total, so the pivot is consistent. Interestingly, Hoos Burgers sold only $171.00 of Drink, compared with $502.50 at Cav Merch North and $582.00 at V-18. Cav Merch North has 'Merch' in its name but made just $400.50 from Merch, less than Rotunda Tacos ($489.00) and V-18 ($508.50).")

category          Drink    Food   Merch  RainGear   Total
vendor_name                                              
Cav Merch North   502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers      171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos     298.5   882.0   489.0     244.5  1914.0
Unknown (V-18)    582.0  1018.5   508.5     240.0  2349.0
Total            1554.0  4293.0  1771.5     901.5  8520.0
The bottom-right($8,520.00) matches the Q1 total, so the pivot is consistent. Interestingly, Hoos Burgers sold only $171.00 of Drink, compared with $502.50 at Cav Merch North and $582.00 at V-18. Cav Merch North has 'Merch' in its name but made just $400.50 from Merch, less than Rotunda Tacos ($489.00) and V-18 ($508.50).


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [11]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**a)** Food brings in most of the revenue at 50.4% (\$4,293.00), so every vendor should make sure they never run out of it. However, I think the bigger opportunities are in the gaps between vendors. Hoos Burgers sold only \$171.00 of Drink, while Cav Merch North sold \$502.50 and V-18 sold \$582.00. Since burgers and drinks go together, Hoos Burgers should try combo-ing a drink with every meal. Cav Merch North should also push its merch harderas it only made \$400.50 from Merch, which is less than Rotunda Tacos (\$489.00) even though merch orders are worth a lot (\$22.42 on average per order versus \$17.46 for Drink). RainGear is only 10.6% of revenue, so vendors probably shouldn't stock much of it unless the forecast calls for rain.

**b)** Q3 is the least trustworthy. It asks which vendor has the highest average order revenue, and the answer (V-01 at \$22.60) sounds definite, but the vendors are separated by about \$2.28 from first to last while each average has a standard error of roughly \$1.6 to \$1.8. With order revenue this spread out (a standard deviation around \$17), the ranking could easily flip with a different set of 400 orders, so I would not tell V-01 it is the best. The unmatched vendor (V-18) is also another weakness- it makes up 27.6% of revenue but has no name in the lookup, so anything I say about it in Q5 and Q6 is on data I can't verify.